# Stock Market Prediction System - Phase 2: Data Cleaning and Pre-processing
**Academic College ML Project SOP**

### Objectives:
1. Clean missing values and anomalous zero entries.
2. Outlier detection using Interquartile Range (IQR).
3. Comprehensive Feature Engineering (Returns, SMAs, Volatility, Momentum, Intraday Ratios).
4. Target generation with strict zero-leakage `shift(-1)`.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download clean dataset
symbol = 'TCS.NS'
raw_df = yf.download(symbol, period='5y', interval='1d', progress=False)
if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df.columns = raw_df.columns.get_level_values(0)

df = raw_df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.dropna(inplace=True)
print('Initial clean shape:', df.shape)

### 1. Technical Indicator Feature Engineering

In [ ]:
# Returns
df['daily_return'] = df['Close'].pct_change()
df['return_3d'] = df['Close'].pct_change(3)
df['return_5d'] = df['Close'].pct_change(5)
df['return_10d'] = df['Close'].pct_change(10)

# Simple Moving Averages
df['SMA_5'] = df['Close'].rolling(window=5).mean()
df['SMA_10'] = df['Close'].rolling(window=10).mean()
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()

# Volatility (Rolling Std of Returns)
df['volatility_5'] = df['daily_return'].rolling(window=5).std()
df['volatility_10'] = df['daily_return'].rolling(window=10).std()
df['volatility_20'] = df['daily_return'].rolling(window=20).std()

# Price Relationships
df['high_low_ratio'] = df['High'] / (df['Low'] + 1e-8)
df['close_open_ratio'] = df['Close'] / (df['Open'] + 1e-8)
df['price_range'] = df['High'] - df['Low']

# Momentum and Volume
df['volume_change'] = df['Volume'].pct_change().fillna(0)
df['momentum_5'] = df['Close'] - df['Close'].shift(5)
df['momentum_10'] = df['Close'] - df['Close'].shift(10)

print('Feature engineering complete. Available features:', df.columns.tolist())

### 2. Feature Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.dropna().corr()
sns.heatmap(corr, cmap='coolwarm', annot=False, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.show()

### 3. Target Variable Construction (Zero-Leakage Shift)
- `target_high = High.shift(-1)`
- `target_low = Low.shift(-1)`
- `direction_target = 1 (Bullish) if Tomorrow Close > Today Close else 0 (Bearish)`
- `signal_target = 1 (BUY >= +1%), 2 (SELL <= -1%), 0 (HOLD)`

The final row is dropped because its future outcome has not occurred yet.

In [ ]:
df['target_high'] = df['High'].shift(-1)
df['target_low'] = df['Low'].shift(-1)
df['target_close'] = df['Close'].shift(-1)

df['direction_target'] = (df['target_close'] > df['Close']).astype(int)

next_ret = (df['target_close'] - df['Close']) / df['Close']
signal = pd.Series(0, index=df.index)
signal[next_ret >= 0.01] = 1 # BUY
signal[next_ret <= -0.01] = 2 # SELL
df['signal_target'] = signal

clean_dataset = df.dropna().copy()
print('Final Clean Dataset Shape for Modeling:', clean_dataset.shape)
print('Direction Distribution:\n', clean_dataset['direction_target'].value_counts())
print('Signal Distribution (0=HOLD, 1=BUY, 2=SELL):\n', clean_dataset['signal_target'].value_counts())